In [ ]:
import tifffile as tiff
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

In [ ]:
home_dir = os.path.expanduser("~")
data_dir = os.path.join(home_dir, "ext_hd_sammy", "data")
project_dir = os.path.join(home_dir, "ext_hd_sammy", "projects")

data_dir_comet = os.path.join(data_dir, "COMET")
data_dir_stomics = os.path.join(data_dir, "stomics")
data_dir_msi = os.path.join(data_dir, 'msi')

msi_glycans_path = os.path.join(data_dir_msi, "Glycan OMEtif files")
he_msi_path = os.path.join(data_dir_msi, "he")
if_comet_path = os.path.join(data_dir_comet, "original")
masks_comet_path = os.path.join(data_dir_comet, "visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples")
stomics_path = os.path.join(data_dir_stomics, "gene_exp")

out = os.path.join(project_dir, 'out')
out_matched_images_stom_comet = os.path.join(out, 'matched_images_stomics_COMET')
os.makedirs(out_matched_images_stom_comet, exist_ok=True)


sub_dir_path_to_stomics_dapi = "03.ssDNA_analysis"

meta_data_file_path = [os.path.join(data_dir,file) for file in os.listdir(data_dir) if file.endswith('.csv')][0]

In [ ]:
stomics_dapi_paths = [os.path.join(stomics_path, d, f'{sub_dir_path_to_stomics_dapi}/ssDNA_{d}_regist.tif') for d in os.listdir(stomics_path) if os.path.isdir(os.path.join(stomics_path, d))]
he_msi_paths = [os.path.join(he_msi_path, d) for d in os.listdir(he_msi_path) if d.endswith('.ndpi')]
if_comet_paths = [os.path.join(if_comet_path, d) for d in os.listdir(if_comet_path) if d.endswith('.tiff')]
visio_comet_mask_paths = [os.path.join(masks_comet_path, d) for d in os.listdir(masks_comet_path) if d.endswith('.tif')]
glycan_paths = [os.path.join(msi_glycans_path, d) for d in os.listdir(msi_glycans_path) if d.endswith('.ome.tif')]

In [ ]:
meta = pd.read_csv(meta_data_file_path)
meta.head(3)

In [ ]:
len(stomics_dapi_paths), len(if_comet_paths), len(he_msi_paths), len(glycan_paths), len(visio_comet_mask_paths)

In [ ]:
def normalize_id(name):
    return name.upper().replace('O', '0').strip()

In [ ]:
### find matches for channel-msi, he-msi, if-comet, dapi-stomics
sample_ids = meta['Sample ID']
chip_ids = meta['Chip ID']

### clean up lists
valid_names = set(sample_ids).union(set(chip_ids))
stomics_dapi_paths = [file for file in stomics_dapi_paths if any(name in file for name in valid_names)]
if_comet_paths = [file for file in if_comet_paths if any(name in file for name in valid_names)]
he_msi_paths = [file for file in he_msi_paths if any(name in file for name in valid_names)]
glycan_paths = [file for file in glycan_paths if any(normalize_id(name) in file for name in valid_names)]
visio_comet_mask_paths = [file for file in visio_comet_mask_paths if any(name in file for name in valid_names)]

In [ ]:
visio_comet_mask_paths

In [ ]:
len(stomics_dapi_paths), len(if_comet_paths), len(he_msi_paths), len(glycan_paths), len(visio_comet_mask_paths)

In [ ]:
### only keep the elements that have a match

linked_samples = list(zip(sample_ids, chip_ids))
[print(sample_id, chip_id) for sample_id, chip_id in linked_samples]



In [ ]:

glycan_paths_filtered = [file for file in glycan_paths if any(normalize_id(sample_id) in file or chip_id in file for sample_id, chip_id in linked_samples)]
if_comet_paths_filtered = [file for file in if_comet_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]
he_msi_paths_filtered = [file for file in he_msi_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]
stomics_dapi_paths_filtered = [file for file in stomics_dapi_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]
visio_comet_mask_paths_filtered = [file for file in visio_comet_mask_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]

In [ ]:
visio_comet_mask_paths_filtered

In [ ]:
s0s_in_if = [name for name in sample_ids if any(name in file_path for file_path in if_comet_paths_filtered)]
s0s_in_if

In [ ]:
s0s_in_visio = [name for name in sample_ids if any(name in file_path for file_path in visio_comet_mask_paths_filtered)]
s0s_in_visio

In [ ]:
s0s_in_he = [name for name in sample_ids if any(name in file_path for file_path in he_msi_paths_filtered)]
s0s_in_he

In [ ]:
s0s_glycans = [name for name in sample_ids if any(normalize_id(name) in file_path for file_path in glycan_paths_filtered)]
s0s_glycans

In [ ]:
s0_shared = set(s0s_in_he).intersection(set(s0s_in_if)).intersection(set(s0s_glycans))
s0_shared

In [ ]:
meta_reduced = meta[meta['Sample ID'].isin(s0_shared)]
meta_reduced

In [ ]:
shared_chips = meta_reduced['Chip ID'].tolist()
chips_in_stomics = [name for name in shared_chips if any(name in file_path for file_path in stomics_dapi_paths_filtered)]
chips_in_stomics

In [ ]:
import sys
import shutil

matched_ids = list(zip(meta_reduced['Sample ID'], meta_reduced['Chip ID']))

for sample_id, chip_id in matched_ids:
    print(sample_id, chip_id)
    
    
    he_matched = [fp for fp in he_msi_paths if os.path.basename(fp) == f'{sample_id}.ndpi']
    comet_matched = [fp for fp in if_comet_paths if os.path.basename(fp) == f'{sample_id}_BS.ome.tiff']
    stomics_matched = [fp for fp in stomics_dapi_paths if os.path.basename(fp) == f'ssDNA_{chip_id}_regist.tif']
    glycan_matched = [fp for fp in glycan_paths if os.path.basename(fp) == f'{normalize_id(sample_id)} ovary manual glycans.ome.tif']
    vis_matched = [fp for fp in visio_comet_mask_paths if os.path.basename(fp) == f'{sample_id}_BS.tif']
    
    ### load ans save only only channel for glycan
    g = tiff.imread(glycan_matched[0])
    g_single_channel = g[0,...]
    
    print(he_matched)
    print(comet_matched)
    print(stomics_matched)
    print(glycan_matched)
    print(vis_matched)
    
    if not os.path.exists(stomics_matched[0]):
        print(f"Stomics DAPI image not found for chip ID {chip_id}. Skipping this sample.")
        continue
    
    ### save each data in a new folder
    
    
    new_folder_path = os.path.join(out_matched_images_stom_comet, f'{sample_id}_{chip_id}')
    os.makedirs(new_folder_path, exist_ok=True)
    
    #tiff.imwrite(os.path.join(new_folder_path, f'{normalize_id(sample_id)}_glycans.ome.tif'), g_single_channel)
    #shutil.copy(he_matched[0], new_folder_path)
    shutil.copy(comet_matched[0], new_folder_path) 
    shutil.copy(stomics_matched[0], new_folder_path)
    shutil.copy(vis_matched[0], new_folder_path)
  
    